In [0]:
# """
# ================================================================================
# Superstore Gold Dimensional ETL Orchestrator
# ================================================================================
# Purpose       : Master orchestrator for building and maintaining all SCD2 
#                 dimension tables in the Gold layer of the Superstore Medallion 
#                 Architecture. Handles reading Silver layer tables, applying 
#                 Slowly Changing Dimension Type 2 (SCD2) logic, merging updates, 
#                 handling soft deletes, collecting metrics, and optimizing tables 
#                 for query performance.

# Design Notes  :
# 1. Supports multiple dimensions configured centrally via `dim_config`.
# 2. Implements full SCD2 processing: compute timeline, merge into Gold, handle
#    current/non-current flags, and maintain audit metrics.
# 3. Uses structured logging with master_run_id and layer_run_id for observability
#    across pipelines.
# 4. Optimizations include partitioning (z-ordering), bucketing metrics collection
#    to prevent driver overload, and vacuuming stale data.

# Usage         :
#     python run_gold_dimensions.py  # Execute all configured dimension pipelines
# ================================================================================
# """

from pyspark.sql import SparkSession
import os, sys, time, yaml, importlib, uuid

# -------------------------------
# Spark Session Initialization
# -------------------------------
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

# -------------------------------
# Environment Variables
# -------------------------------
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Add shared utilities path
# -------------------------------
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_gold/core")

# -------------------------------
# Imports
# -------------------------------
from superstore_logger import get_superstore_logger, log_event
from superstore_orchestrator_execution_context import get_execution_context
from superstore_platform_constants import GOLD_LAYER
from superstore_platform_config import get_silver_schema, get_gold_schema, get_metrics_schema, table

# -------------------------------
# Execution context
# -------------------------------
execution_context = get_execution_context(dbutils, allow_standalone=True)
master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]

logger_dim_orchestrator = get_superstore_logger("superstore_gold_dimensional_orchestrator")

log_event(
    logger_dim_orchestrator,
    "INFO",
    "Initiating Superstore Gold layer ETL dimensional orchestrator",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=GOLD_LAYER
)

# -------------------------------
# Load dimension config
# -------------------------------
config_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/configs/superstore_gold_config/superstore_gold_dimension_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
dim_config = config["dim_config"]

# -------------------------------
# Import dimension framework
# -------------------------------
import superstore_gold_dimension_framework
importlib.reload(superstore_gold_dimension_framework)
from superstore_gold_dimension_framework import (
    read_silver_table,
    prepare_scd2_columns,
    compute_scd2_timeline,
    create_gold_table_if_not_exists,
    merge_into_gold_table_scd2,
    handle_soft_deletes,
    collect_metrics,
    optimize_gold_table,
    vacuum_gold_table,
)

# -------------------------------
# Gold Dimensional ETL Orchestrator
# -------------------------------
def run_all_dim_pipelines(spark, master_run_id: str):
    log_event(
        logger_dim_orchestrator,
        "INFO",
        "Starting Gold dimensional pipelines for all configured tables",
        total_tables=len(dim_config),
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )

    for table_name, cfg in dim_config.items():

        # -----------------------------
        # Dynamically resolve full catalog.schema.table
        # -----------------------------
        silver_tbl = table(get_silver_schema(), cfg["silver_tbl"])
        gold_tbl = table(get_gold_schema(), cfg["gold_tbl"])
        metrics_table = table(get_metrics_schema(), cfg["metrics_table"])
    
        layer_name = cfg.get("layer_name")
        table_type = cfg.get("table_type")
        required_columns = cfg["required_columns"]
        natural_keys = cfg["natural_keys"]
        column_mapping = cfg["column_mapping"]
        hash_column = cfg["hash_column"]
        silver_hash_column = cfg["silver_hash_column"]
        z_order_cols = cfg.get("z_order_cols", [])
        max_rows_per_bucket = cfg.get("max_rows_per_bucket", 1000000)
        entity_id_col = natural_keys[0]  # Assumes single natural key for SCD2

        start_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
        start_time = time.time()
        run_status = "SUCCESS"
        notes = ""
        has_incremental_rows = False

        try:
            log_event(
                logger_dim_orchestrator,
                "INFO",
                f"Starting dimensional pipeline for {gold_tbl}",
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 1: Read Silver table
            df_silver = read_silver_table(
                spark=spark,
                silver_table=silver_tbl,
                gold_table=gold_tbl,
                required_columns=required_columns,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            has_incremental_rows = df_silver.limit(1).count()
            if not has_incremental_rows:
                log_event(
                    logger_dim_orchestrator,
                    "INFO",
                    f"No new rows to process for {gold_tbl}; skipping",
                    table=gold_tbl,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )
                collect_metrics(
                    spark,
                    repaired_df=df_silver,
                    gold_df=spark.table(gold_tbl) if spark.catalog.tableExists(gold_tbl) else None,
                    gold_tbl=gold_tbl,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    table_name=gold_tbl,
                    table_type=table_type,
                    entity_id_column=entity_id_col,
                    silver_hash_column=silver_hash_column,
                    gold_hash_column=hash_column,
                    metrics_table=metrics_table,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0,
                    run_status="skipped",
                    max_rows_per_bucket=max_rows_per_bucket,
                    notes="No new rows or files to process"
                )
                continue

            # Step 2: Prepare SCD2 columns
            df_scd2 = prepare_scd2_columns(
                spark=spark,
                silver_df=df_silver,
                entity_columns=required_columns,
                hash_column=hash_column,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 3: Compute timeline
            df_timeline = compute_scd2_timeline(
                scd_df=df_scd2,
                entity_id_column=entity_id_col,
                hash_column=hash_column,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 4: Ensure Gold table exists
            create_gold_table_if_not_exists(
                spark=spark,
                repaired_df=df_timeline,
                gold_tbl=gold_tbl,
                partition_column=z_order_cols[0] if z_order_cols else None,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 5: Merge timeline into Gold (SCD2)
            merge_into_gold_table_scd2(
                spark=spark,
                repaired_df=df_timeline,
                gold_tbl=gold_tbl,
                entity_id_column=entity_id_col,
                hash_column=hash_column,
                max_rows_per_bucket=max_rows_per_bucket,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )

            # Step 6: Handle soft deletes
            handle_soft_deletes(
                spark=spark,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER,
                silver_df=df_silver,
                gold_tbl=gold_tbl,
                entity_id_column=entity_id_col,
            )

        except Exception as e:
            run_status = "FAILED"
            notes = str(e)
            log_event(
                logger_dim_orchestrator,
                "ERROR",
                f"Dimensional pipeline failed for table {gold_tbl}: {e}",
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=GOLD_LAYER
            )
            raise

        finally:
            if has_incremental_rows:
                end_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
                duration_secs = int(time.time() - start_time)

                collect_metrics(
                    spark,
                    repaired_df=df_silver,
                    gold_df=spark.table(gold_tbl),
                    gold_tbl=gold_tbl,
                    layer_name=layer_name,
                    silver_table=silver_tbl,
                    table_name=gold_tbl,
                    table_type=table_type,
                    entity_id_column=entity_id_col,
                    silver_hash_column=silver_hash_column,
                    gold_hash_column=hash_column,
                    metrics_table=metrics_table,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER,
                    start_ts=start_ts,
                    end_ts=end_ts,
                    duration_secs=duration_secs,
                    run_status=run_status,
                    max_rows_per_bucket=max_rows_per_bucket,
                    notes=notes
                )

                optimize_gold_table(
                    spark,
                    gold_tbl=gold_tbl,
                    hash_column=hash_column,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                vacuum_gold_table(
                    spark,
                    gold_tbl=gold_tbl,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

                log_event(
                    logger_dim_orchestrator,
                    "INFO",
                    f"Completed dimensional pipeline for table: {gold_tbl}",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=GOLD_LAYER
                )

    log_event(
        logger_dim_orchestrator,
        "INFO",
        "Completed all Gold dimensional pipelines",
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=GOLD_LAYER
    )

# -------------------------------
# Run ETL
# -------------------------------
if __name__ == "__main__":
    run_all_dim_pipelines(spark, master_run_id)
